In [0]:
#Esto carga tablas finales

import pandas as pd
import numpy as np

model_results = spark.table("workspace.gold.model_results_all").toPandas()
match_predictions = spark.table("workspace.gold.worldcup_2026_match_predictions").toPandas()
montecarlo_summary = spark.table("workspace.gold.worldcup_2026_montecarlo_summary").toPandas()

print("Model results:", model_results.shape)
print("Match predictions:", match_predictions.shape)
print("Monte Carlo summary:", montecarlo_summary.shape)

In [0]:
#compara modelos

display(
    model_results
    .sort_values("log_loss", ascending=True)
)

In [0]:

#top 20 favoritos a campeon 
top_champions = (
    montecarlo_summary
    .sort_values("prob_champion", ascending=False)
    .head(20)
)

display(top_champions)

In [0]:

#top favoritos con porcentaje 

top_champions_view = top_champions.copy()

percentage_cols = [
    "prob_round_32",
    "prob_round_16",
    "prob_quarterfinal",
    "prob_semifinal",
    "prob_final",
    "prob_champion"
]

for col in percentage_cols:
    top_champions_view[col] = (top_champions_view[col] * 100).round(2)

display(top_champions_view)

In [0]:

#grafica campeon 

import matplotlib.pyplot as plt

chart_data = top_champions.sort_values("prob_champion", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(chart_data["team"], chart_data["prob_champion"] * 100)
plt.xlabel("Probabilidad de campeón (%)")
plt.ylabel("Selección")
plt.title("Top 20 selecciones con mayor probabilidad de ganar el Mundial 2026")
plt.tight_layout()
plt.show()

In [0]:


#Probabilidad por ronda top 10 
top_10 = (
    montecarlo_summary
    .sort_values("prob_champion", ascending=False)
    .head(10)
    .copy()
)

round_cols = [
    "prob_round_32",
    "prob_round_16",
    "prob_quarterfinal",
    "prob_semifinal",
    "prob_final",
    "prob_champion"
]

top_10_percent = top_10[["team"] + round_cols].copy()

for col in round_cols:
    top_10_percent[col] = (top_10_percent[col] * 100).round(2)

display(top_10_percent)

In [0]:
#partido con favoritos mas fuertes 

match_predictions["max_probability"] = match_predictions[
    ["prob_home_win", "prob_draw", "prob_away_win"]
].max(axis=1)

strong_favorites = (
    match_predictions
    .sort_values("max_probability", ascending=False)
    .head(20)
)

display(strong_favorites[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "predicted_result",
    "max_probability"
]])

In [0]:

#partidos mas parejos 
balanced_matches = (
    match_predictions
    .sort_values("max_probability", ascending=True)
    .head(20)
)

display(balanced_matches[[
    "home_team",
    "away_team",
    "fifa_rank_home",
    "fifa_rank_away",
    "fifa_points_home",
    "fifa_points_away",
    "prob_home_win",
    "prob_draw",
    "prob_away_win",
    "predicted_result",
    "max_probability"
]])

In [0]:
team_fifa_info = {}

for _, row in match_predictions.iterrows():
    team_fifa_info[row["home_team"]] = {
        "fifa_rank": row["fifa_rank_home"],
        "fifa_points": row["fifa_points_home"]
    }
    team_fifa_info[row["away_team"]] = {
        "fifa_rank": row["fifa_rank_away"],
        "fifa_points": row["fifa_points_away"]
    }

montecarlo_summary["fifa_rank"] = montecarlo_summary["team"].map(
    lambda team: team_fifa_info.get(team, {}).get("fifa_rank")
)

montecarlo_summary["fifa_points"] = montecarlo_summary["team"].map(
    lambda team: team_fifa_info.get(team, {}).get("fifa_points")
)

display(montecarlo_summary.head())

In [0]:
display(
    montecarlo_summary[
        montecarlo_summary["team"] == "Colombia"
    ][[
        "team",
        "fifa_rank",
        "fifa_points",
        "prob_round_32",
        "prob_round_16",
        "prob_quarterfinal",
        "prob_semifinal",
        "prob_final",
        "prob_champion"
    ]]
)

In [0]:
display(
    match_predictions[
        (match_predictions["home_team"] == "Colombia") |
        (match_predictions["away_team"] == "Colombia")
    ][[
        "date",
        "home_team",
        "away_team",
        "fifa_rank_home",
        "fifa_rank_away",
        "fifa_points_home",
        "fifa_points_away",
        "prob_home_win",
        "prob_draw",
        "prob_away_win",
        "predicted_result"
    ]]
)

# Conclusiones

El proyecto construye un modelo predictivo para estimar resultados del Mundial 2026 usando datos históricos de partidos internacionales, ranking FIFA, forma reciente, goles recientes, contexto del torneo y simulación Monte Carlo.

## Principales resultados

- Se entrenaron modelos base y una red neuronal MLP.
- Se integró ranking FIFA histórico para entrenamiento.
- Para la simulación 2026 se usó un ranking FIFA oficial corregido.
- Se generaron predicciones para los 72 partidos de fase de grupos.
- Se ejecutaron 5.000 simulaciones Monte Carlo para estimar probabilidades de avance y campeón.

## Limitaciones

- El bracket de eliminatorias es simplificado.
- La simulación usa probabilidades de resultado, no marcador exacto.
- No se modelan lesiones, convocatorias, entrenadores ni cambios tácticos.
- La probabilidad de penales se maneja como 50/50 cuando hay empate en eliminatorias.

## Mejoras futuras

- Incorporar bracket oficial completo del Mundial 2026.
- Modelar goles esperados para desempates de grupo.
- Agregar variables de continente, sede y distancia de viaje.
- Comparar con Elo externo.
- Crear dashboard en Power BI o Databricks.